# Web Scraping

Justin Chun-ting Ho

Last Update: 2026-06-18


Leanring goals:
1. How a browser requests a webpage from a web server
2. How HTML stores page content
3. How CSS selectors help locate specific elements
4. How Beautiful Soup extracts text from HTML
5. How to store scraped data in a table

## 1. Install and import libraries

Run this installation cell if your environment does not already have the packages.

In [ ]:
# Install packages if needed
# You can skip this cell if these packages are already installed.

%pip install requests beautifulsoup4 pandas lxml

In [ ]:
# Import libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

## 2. Download the webpage

A webpage is sent from a web server to your computer as HTML.

`requests.get()` asks the server for the page.

`"User-Agent": "Mozilla/5.0"` tells the server who you are.

In [ ]:
response = requests.get("https://dcat.nycu.edu.tw/members/faculty-and-staff/",
                        timeout=5.0,
                        headers={"User-Agent": "Mozilla/5.0"})

print(response.status_code)

A status code of `200` usually means the request was successful. Here's a list for others:

| Code | Meaning | Use |
|---:|---|---|
| 200 | OK | Request succeeded |
| 201 | Created | Resource created |
| 204 | No Content | Success, no response body |
| 301 | Moved Permanently | Permanent redirect |
| 302 | Found | Temporary redirect |
| 400 | Bad Request | Invalid request |
| 401 | Unauthorized | Login/auth required |
| 403 | Forbidden | Not allowed |
| 404 | Not Found | Resource missing |
| 429 | Too Many Requests | Rate limited |
| 500 | Internal Server Error | Server error |
| 502 | Bad Gateway | Bad upstream response |
| 503 | Service Unavailable | Server overloaded/down |
| 504 | Gateway Timeout | Upstream timeout |

Now look at the first part of the HTML.

In [ ]:
print(response.text)

## 3. Parse the HTML with Beautiful Soup

Raw HTML is just a long string.

Beautiful Soup turns that string into a structure we can search.

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

The page title is a simple first check that Beautiful Soup is working.

In [ ]:
soup.title

## 4. Inspect the HTML

On this page, a member name can appear inside an HTML element like this:

```html
<h5 class="brxe-ydknxx brxe-post-title member-list-name">賴至慧(系主任)</h5>
```

Important parts:

- `h5` is the tag name
- `class="... member-list-name"` marks the element as a member name
- `賴至慧(系主任)` is the visible text

In Beautiful Soup, `.member-list-name` means: find elements with class `member-list-name`.

In [ ]:
# Find the first member name element

first_name_tag = soup.select_one(".member-list-name")

print(first_name_tag)

In [ ]:
# Extract only the visible text

first_name_tag.get_text()

## 5. Find all member name elements

Now that one element works, we can find all elements with the same class.

In [ ]:
name_tags = soup.select(".member-list-name")

In [ ]:
names = []
for tag in name_tags:
    names.append(tag.get_text())
names

## 6. Understand the surrounding structure

A common scraping strategy is:

1. Find the target element
2. Look at nearby HTML
3. Decide where the related information is stored

Here, we first inspect the parent area around one name.

In [ ]:
# Look at the surrounding HTML for the first member

first_member_area = first_name_tag.find_parent()

print(first_member_area.prettify()[:2000])

## 7. Extract profile links

Some names are inside links. A link is stored in an `<a>` tag.

The `href` attribute usually stores the profile URL.

Use `get()` to get it.

In [ ]:
# Solving one element
name_tags[1].find_parent().get("href")

In [ ]:
# Applying to everything, getting all links
links = []
for name_tag in name_tags:
    new_link = name_tag.find_parent().get("href")
    links.append(new_link)

print(links)

In [ ]:
# Make a dataframe
df = pd.DataFrame({'name': names, 'link': links})
df

## 8. Crawling

Sometimes information are stored inside separate pages. We can build a crawler to visit one by one.

To get the email, we can use this spell: `soup.select('a[href^="mailto:"]')`

| Part         | Meaning                                         |
| ------------ | ----------------------------------------------- |
| `a`          | Select `<a>` hyperlink tags                     |
| `[href ...]` | Only select links that have an `href` attribute |
| `^=`         | “starts with”                                   |
| `"mailto:"`  | The starting text we are looking for            |

You could look up the syntax, or even learn Regular Expression. But to be honest it is easier to just ask ChatGPT nowadays:

```Write a python for loop, that will visit each link, extract the texts after mailto:```

In [ ]:
response = requests.get(links[0], timeout=5.0,
                        headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
soup.select_one('a[href^="mailto:"]')#.get("href")#.replace("mailto:", "")

In [ ]:
for link in links[0:3]:
    response = requests.get(link,
                        timeout=5.0,
                        headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    # Find all email links on the page
    tag = soup.select_one('a[href^="mailto:"]')
    email = tag.get("href")
    print(email)

# A Realistic Example

Let's get some news data from: https://news.ltn.com.tw

In [ ]:
response = requests.get("https://news.ltn.com.tw/list/breakingnews/politics",
                        timeout=5.0,
                        headers={"User-Agent": "Mozilla/5.0"})
print(response.status_code)

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
soup.select_one('.title')#.find_parent()#.find_parent()#.get('href')

In [ ]:
# After we can extract one, we can extract all
tags = soup.select('.title')
links = []
for tag in tags:
    new_link = tag.find_parent().find_parent().get("href")
    links.append(new_link)
links

## Get one article

In [ ]:
response = requests.get("https://news.ltn.com.tw/news/politics/breakingnews/5476751",
                        timeout=5.0,
                        headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
soup.select_one('h1')

In [ ]:
soup.select('p')

## A Playwright Example to Deal with Procedural Generation

In [2]:
from playwright.async_api import async_playwright

url = "https://news.ltn.com.tw/list/breakingnews/politics"

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    page = await browser.new_page()

    await page.goto(url, wait_until="domcontentloaded", timeout=10000)

    # Scroll several times to trigger more articles to load
    for i in range(5):
        await page.mouse.wheel(0, 3000)
        await page.wait_for_timeout(1500)

    # Collect article links after scrolling
    links = await page.locator("a").evaluate_all(
        """
        elements => elements
            .map(a => ({
                text: a.innerText.trim(),
                href: a.href
            }))
            .filter(item =>
                item.text.length > 0 &&
                item.href.includes('/news/politics/breakingnews/')
            )
        """
    )

    await browser.close()

links[:10]

[{'text': '獨家》台南市長選舉難整合？陳明文南下 促妃憲派議員大團結',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481494'},
 {'text': '廢監院主張與鄭麗文不同調？ 蔣萬安：尊重黨團自主',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481451'},
 {'text': 'AIT谷立言看好台灣無人機 台中7/2舉辦論壇鏈結國際供應鏈',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481502'},
 {'text': '蘇巧慧協調經濟部拆瑞芳蝙蝠洞禁潛告示牌 盼兼顧漁業永續、觀光發展',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481526'},
 {'text': '中國「民族團結進步促進法」7月施行 台派：破壞國際法、台有權捍衛人權自由',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481525'},
 {'text': '又被網抓包背稿！ 蔣萬安還背到重複上秒才說過的話......',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481452'},
 {'text': '陳瑩問要買多少鳳梨釋迦？蔣萬安反嗆：2023就行銷不是選舉才做',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481512'},
 {'text': '與銀髮選手切磋《英雄聯盟》 李四川：爭取設立新北第二座電競基地',
  'href': 'https://news.ltn.com.tw/news/politics/breakingnews/5481511'},
 {'text': '沈伯洋拋臨托政見 蔣萬安：任內據點成長逾10倍',
  'href': 'https://news.l